# Module 03 — Product structural change
### Part 1: model mix and electrification

## ⚠️ Why this module is split

The thesis's distinctive contribution is bringing **vehicle size** into a structural-change reading of employment. That needs per-model mass — and the attribute source in the dataset cannot support it.

`tbl_ademe_variants` holds **1,890 rows / 80 distinct (brand, model) pairs across 18 brands**. For Mercedes it contains only *E 200/300/400/E 53 AMG*; for Audi only *A3, e-tron GT, Q3*; for VW only *Caddy, Touran*.

Measured consequences:

| | |
|---|---|
| plant-model-years with attributes | **15%** |
| model mix covered in the balanced German universe | **12%** |
| Mercedes match rate | **1 / 137** |
| VW / Audi | 3/39 · 8/53 |
| Porsche / Renault | 12/22 · 18/37 |

The unmatched set is the **most-produced cars in Europe** — Golf, C-Class, Passat, Polo, Tiguan, Sprinter, Cayenne. A mean kerb weight on this sample would track *which brands happened to match*, not the production mix: it could show weight **falling** for a plant that merely swapped an unmatched Golf for a matched Polo.

**So mass-based analysis is deferred to Module 03b**, once the [EEA CO2 monitoring database](https://www.eea.europa.eu/en/datahub/datahubitem-view/fa8b1229-3db6-495d-b18e-9c9b3267c02b) (Reg. (EU) 2019/631 — EU-wide, per-registration, **mass in running order**, fuel type, make and commercial name, 2010→) replaces ADEME.

This module does everything that does **not** depend on mass.

In [ ]:
# Reload edited .py modules automatically — without this, a kernel that
# imported an older version keeps serving it (AttributeError on new functions).
%load_ext autoreload
%autoreload 2

import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import foundation as fd, module02_organisational as m2, module03_product as m3
# --- archive bootstrap: resolve paths from the project root ---------------
# The original was CONFIG.data_dir = './data', which assumed the notebook ran
# beside a data/ folder. In the archived layout the inputs live in
# 02_data_inputs/, so the path is taken from specular_paths instead and this
# notebook works from wherever it is opened.
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parent))   # 01_code/
from specular_paths import P
fd.CONFIG.data_dir = str(P.inputs)
fd.CONFIG.out_dir  = str(P.outputs)
print('data:', fd.CONFIG.data_dir)
fd.CONFIG.out_dir  = './outputs'
pd.set_option('display.width', 200)

## 1. The attribute-readiness diagnostic

This belongs in the **methods chapter** — it is the evidence for switching source. Panel (c) sizes the task: attributes for the top **150** models by frequency would cover **61%** of plant-model-years, top **200** → **71%**, top **300** → **86%**.

In [ ]:
ar = m3.attribute_readiness()
fig = m3.fig_attribute_readiness(ar, save=str(fd.CONFIG.out('fig03_attribute_readiness.png'))); plt.show()
display(ar['by_brand'][ar['by_brand']['models']>=5].round(0))

## 2. Electrification without attributes

ADEME's fuel field reaches 15% of the mix. But **dedicated BEV nameplates are identifiable by name** (ID.3, i4, EQS, e-tron, Mégane E-Tech…) at near-full coverage. The classifier is hand-verified against every candidate name in the panel.

The traps it must avoid are real and present in this data:

| name | reality |
|---|---|
| `C-Class EQ`, `GLC mit EQ-Technologie` | EQ **mild hybrid**, not BEV |
| `Punto EVO` | *evoluzione* — nothing electric |
| `Macan S`, `Macan GTS`, … | ICE; only `Macan 4 Electric` is BEV |
| `Electric Vehicle Drive Modules` | a **component**, not a car |

In [ ]:
for t in ['C-Class EQ','GLC mit EQ-Technologie','Punto EVO','Macan S','Auris Hybrid','Electric Vehicle Drive Modules']:
    assert not m3.classify_bev(t), t
for t in ['ID.3','i4 eDrive40','EQS','Q4 e-tron','e-Golf','Macan 4 Electric','Mégane E-Tech','Born']:
    assert m3.classify_bev(t), t
print('all spot-checks pass')

_ = m3.validate_bev_classifier()

### 2b. What the validation actually shows

Precision on the overlap is 71%, and **both disagreements are ADEME matching errors, not classifier errors**:

* *Porsche Macan 4 Electric* → ADEME says **Petrol** (it matched the ICE Macan)
* *Ford Explorer EV* → ADEME says **Petrol plug-in hybrid** (it matched the US PHEV, not the Cologne-built BEV)

That is further evidence the ADEME layer is unreliable — and a reason to trust the nameplate route for the EV axis.

### 2c. A back-projection bug the guard catches

The source data puts the **Mercedes EQA at Rastatt in 2006–2012**. The EQA launched in **2021** — the press gap-filler assigned the plant's *current* model list to earlier years. Same for the BMW iX1 in 2013 (launched 2022).

`BEV_FIRST_YEAR` flags any BEV record pre-dating its nameplate's launch and sets `is_bev=False`, so the early electrification series is not fictitious. Records are **flagged, not deleted** (`bev_implausible`).

In [ ]:
mp = m3.model_panel()
print('implausible BEV records neutralised:', int(mp['bev_implausible'].sum()))
display(mp[mp['bev_implausible']][['plant_name','year','model','value_source']].head())
print('\nfirst BEV year after the guard:', int(mp[mp['is_bev']]['year'].min()), '(BMW i3 — correct)')

In [ ]:
fig = m3.fig_electrification(mp, save=str(fd.CONFIG.out('fig03_electrification.png'))); plt.show()
g = mp.groupby('year')['is_bev'].agg(['sum','size']); g['bev_pct'] = (100*g['sum']/g['size']).round(1)
display(g.tail(12))

## 3. Model mix and product-cycle dynamics

Portfolio size, churn (models added/dropped), renewal rate — the product-cycle pulse of the network.

In [ ]:
mix = m3.plant_model_mix(mp)
fig = m3.fig_model_mix(mix, save=str(fd.CONFIG.out('fig03_model_mix.png'))); plt.show()
display(mix.groupby('year')[['n_models','n_bev','bev_share']].mean().round(2).tail(8))

In [ ]:
ch = m3.model_churn(mp)
fig = m3.fig_churn(ch, save=str(fd.CONFIG.out('fig03_churn.png'))); plt.show()
print('most-churned plants (total model entries):')
display(ch.groupby(['plant_name','country_iso2'])['entries'].sum().sort_values(ascending=False).head(10))
ch.to_csv('outputs/tbl03_model_churn.csv', index=False)

## 4. Multi-sourcing

How many plants build a given model at once — a footprint-strategy indicator. Rising values mean flexibility, or a transfer in progress.

In [ ]:
ms = m3.multi_sourcing(mp)
fig = m3.fig_multi_sourcing(ms, save=str(fd.CONFIG.out('fig03_multi_sourcing.png'))); plt.show()
print('most multi-sourced models (max plants at once):')
display(ms.groupby('model')['n_plants'].max().sort_values(ascending=False).head(10))

## 5. Employment beside the model mix — descriptive only

On the strictly balanced universe (31 plants, 2012–2023). These are **associations**, not effects; nothing here identifies causality and the thesis does not claim it.

Note especially panel (c): it compares two **changing** sets of plants — a plant moves between them the year it starts building a BEV. Read it as composition, not as an effect of electrification.

In [ ]:
ev = m3.employment_vs_mix(window=(2012, 2023))
fig = m3.fig_employment_vs_mix(ev, save=str(fd.CONFIG.out('fig03_employment_vs_mix.png'))); plt.show()
print('correlation (plant-years, descriptive):')
print(ev[['employees','n_models','n_bev','bev_share']].corr().round(2).to_string())

---
## What Module 03 establishes

* **BEV share of the model mix rises from 0% (pre-2013) to 9.3% (2025)**, starting with the BMW i3 in 2013 — measured at near-full coverage from nameplates, where the attribute route would have reached 15%.
* **Portfolio size grew** from ~4.5 to ~8.3 models per plant: plants build more distinct models than they did.
* Model **entries consistently exceed exits** — the network is adding nameplates, not just rotating them.
* The **attribute-readiness diagnostic** is a methods-chapter figure in its own right, and it quantifies the target for the replacement source.

## Open items

1. **Module 03b — mass.** Download the EEA CO2 monitoring database, match on make + commercial name, and redo the size analysis. Target ≥200 models (≈71% of plant-model-years).
2. **Churn is partly a coverage artefact.** A model 'enters' when it first appears in that plant's sources. Worth restricting churn analysis to well-covered plant-years before drawing conclusions.
3. **`BEV_FIRST_YEAR` should be extended** as more BEV nameplates enter the panel; it currently covers those present.
4. The **back-projection bug** in `established_press` may affect non-BEV models too — worth a systematic check of model start-years against launch dates.

**Next — Module 04: macro–micro integration**, using the plant→NUTS crosswalk to place plants inside their regional context and measure what share of each region's NACE 29 employment they represent.